In [3]:
# ============================================================
# FINE-TUNING NO GOOGLE COLAB — TECH CHALLENGE FASE 3
# ============================================================

# CÉLULA 1 — Instalação
!pip install unsloth datasets -q
!pip install torch torchvision torchaudio -q

import torch
print(f"✅ CUDA disponível: {torch.cuda.is_available()}")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [9]:
# CÉLULA 2 — Prepara os dados direto no Colab
import json
from datasets import load_dataset
from pathlib import Path

Path("data").mkdir(exist_ok=True)

# Carrega os dois datasets
print("⏳ Baixando PubMedQA...")
ds_pubmed  = load_dataset("qiaojin/PubMedQA",  "pqa_labeled", trust_remote_code=True)
print("⏳ Baixando MedQuAD...")
ds_medquad = load_dataset("lavita/MedQuAD", trust_remote_code=True)


TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{output}<|end_of_text|>"""

def formatar(inst, inp, out):
    return TEMPLATE.format(instruction=inst, input=inp, output=out)

dados = []

# PubMedQA
for i, item in enumerate(ds_pubmed['train']):
    if i >= 500: break
    if len(item['long_answer'].strip()) < 50: continue
    dados.append(formatar(
    "Você é um assistente médico especializado. Responda SEMPRE em português brasileiro, com base em evidências científicas.",
    item['question'].strip(),
    f"{item['long_answer'].strip()} [Conclusão: {item['final_decision']}]"
))

# MedQuAD
for i, item in enumerate(ds_medquad['train']):
    if i >= 500: break
    q = str(item.get('question','')).strip()
    a = str(item.get('answer','')).strip()
    if len(q) < 10 or len(a) < 50 or a.lower() == 'none': continue
    dados.append(formatar(
    "Você é um assistente médico. Responda SEMPRE em português brasileiro, de forma clara e precisa.",
    q, a
))

print(f"✅ {len(dados)} exemplos preparados")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⏳ Baixando PubMedQA...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lavita/MedQuAD' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lavita/MedQuAD' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⏳ Baixando MedQuAD...
✅ 1000 exemplos preparados


In [11]:
# CÉLULA 3 — Fine-tuning com QLoRA
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import time

# Carrega modelo
modelo, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    load_in_4bit   = True,
)

# Adiciona adaptadores LoRA
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

# Split treino/validação
split = int(len(dados) * 0.9)
ds_train = Dataset.from_dict({"text": dados[:split]})
ds_val   = Dataset.from_dict({"text": dados[split:]})

# Treinamento
trainer = SFTTrainer(
    model              = modelo,
    tokenizer          = tokenizer,
    train_dataset      = ds_train,
    eval_dataset       = ds_val,
    dataset_text_field = "text",
    max_seq_length     = 2048,
    args = TrainingArguments(
        output_dir                  = "checkpoints",
        num_train_epochs            = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 50,
        learning_rate               = 2e-4,
        fp16                        = True,
        logging_steps               = 25,
        eval_strategy               = "steps",
        eval_steps                  = 100,
        save_strategy               = "steps",
        save_steps                  = 100,
        load_best_model_at_end      = True,
        optim                       = "adamw_8bit",
        seed                        = 42,
        report_to                   = "none",
    ),
)

print("🏋️ Treinando... (~20 minutos no T4)\n")
inicio = time.time()
resultado = trainer.train()
print(f"\n✅ Concluído em {(time.time()-inicio)/60:.1f} min")
print(f"   Loss final: {resultado.training_loss:.4f}")

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/900 [00:00<?, ? examples/s]

...

🏋️ Treinando... (~20 minutos no T4)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 3 | Total steps = 339
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.356308,1.164261
200,1.126216,1.103272
300,1.046269,1.113913
339,1.090652,1.112506



✅ Concluído em 9.2 min
   Loss final: 1.3427


In [12]:
# CÉLULA 4 — Testa e salva o modelo (respostas em português)
FastLanguageModel.for_inference(modelo)

def testar(pergunta):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Você é um assistente médico especializado. Responda SEMPRE em português brasileiro,
de forma clara, objetiva e baseada em evidências científicas.<|eot_id|><|start_header_id|>user<|end_header_id|>
{pergunta}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = modelo.generate(
            **inputs,
            max_new_tokens  = 300,
            temperature     = 0.3,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0], skip_special_tokens=True)
    return resp.split("assistant")[-1].strip()

print("🏥 TESTES DO MODELO — Respostas em Português\n")
perguntas = [
    "Quais são os principais fatores de risco para o câncer de mama?",
    "Qual é o tratamento recomendado para diabetes tipo 2?",
    "Como a hipertensão afeta a saúde cardiovascular?",
]
for q in perguntas:
    print(f"❓ {q}")
    print(f"🤖 {testar(q)[:400]}...")
    print("-"*60)

# Salva os adaptadores
modelo.save_pretrained("modelo_medico_lora")
tokenizer.save_pretrained("modelo_medico_lora")
print("✅ Modelo salvo!")

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🏥 TESTES DO MODELO — Respostas em Português

❓ Quais são os principais fatores de risco para o câncer de mama?


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 The risk of breast cancer is influenced by several factors. These include age, family history, and the presence of other breast cancer risk factors. [Conclusão: yes]...
------------------------------------------------------------
❓ Qual é o tratamento recomendado para diabetes tipo 2?


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 The treatment of type 2 diabetes should be based on the patient's overall health status and the presence of comorbidities. [Conclusão: yes]...
------------------------------------------------------------
❓ Como a hipertensão afeta a saúde cardiovascular?
🤖 The cardiovascular system is affected by hypertension. The pathophysiological mechanisms of hypertension are complex and multifactorial. The cardiovascular system is affected by hypertension through the activation of the renin-angiotensin-aldosterone system (RAAS), which is involved in the regulation of blood pressure. The RAAS is activated in response to the activation of the kidneys by angiotens...
------------------------------------------------------------
✅ Modelo salvo!


In [13]:
# CÉLULA 5 — Baixa o modelo para sua máquina local
# Comprime a pasta e baixa pelo Colab

import shutil
shutil.make_archive("modelo_medico_lora", "zip", "modelo_medico_lora")

from google.colab import files
files.download("modelo_medico_lora.zip")
print("✅ Download iniciado!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado!
